In [0]:
WITH base AS (
  SELECT
      'webos24' AS platform,
      date_ym,
      CAST(log_create_time AS TIMESTAMP) AS log_create_time,
      mac_addr,
      X_User_Number,
      X_Device_Country,
      X_Device_Sales_Model,
      DEVICE_NAME,
      CAST(normal_log:visible AS BOOLEAN) AS visible,
      CAST(normal_log:app_id AS STRING) AS app_id
  FROM kic_data_ods.tlamp.normal_log_webos24
  WHERE 1=1
    AND date_ym = '2025-05'
    AND log_create_time >= '2025-05-01'
    AND X_Device_Country = 'KR'
    AND (
      X_Device_Sales_Model RLIKE '^OLED[0-9]{2}(G4|C4|B4).*'
      OR X_Device_Sales_Model LIKE '%QNED80%'
    )
    AND context_name = 'LSM'
    AND (
      normal_log:app_id IN (
        'netflix',
        'com.disney.disneyplus-prod',
        'youtube.leanback.v4',
        'amazon',
        'com.webos.app.lgchannels',
        'com.webos.app.home',
        'com.webos.app.livetv'
      )
      OR normal_log:app_id LIKE 'com.webos.app.hdmi%'
    )

  UNION ALL

  SELECT
      'webos23' AS platform,
      date_ym,
      CAST(log_create_time AS TIMESTAMP) AS log_create_time,
      mac_addr,
      X_User_Number,
      X_Device_Country,
      X_Device_Sales_Model,
      DEVICE_NAME,
      CAST(normal_log:visible AS BOOLEAN) AS visible,
      CAST(normal_log:app_id AS STRING) AS app_id
  FROM kic_data_ods.tlamp.normal_log_webos23
  WHERE 1=1
    AND date_ym = '2025-05'
    AND log_create_time >= '2025-05-01'
    AND X_Device_Country = 'KR'
    AND (
      X_Device_Sales_Model RLIKE '^OLED[0-9]{2}(C3|B3).*'
      OR X_Device_Sales_Model LIKE '%UR8300%'
    )
    AND context_name = 'LSM'
    AND (
      normal_log:app_id IN (
        'netflix',
        'com.disney.disneyplus-prod',
        'youtube.leanback.v4',
        'amazon',
        'com.webos.app.lgchannels',
        'com.webos.app.home',
        'com.webos.app.livetv'
      )
      OR normal_log:app_id LIKE 'com.webos.app.hdmi%'
    )
),

ordered AS (
  SELECT
      *,
      LAG(log_create_time) OVER (
        PARTITION BY platform, mac_addr, app_id
        ORDER BY log_create_time
      ) AS prev_log_create_time,
      LAG(visible) OVER (
        PARTITION BY platform, mac_addr, app_id
        ORDER BY log_create_time
      ) AS prev_visible
  FROM base
),

durations_raw AS (
  SELECT
      platform,
      mac_addr,
      X_User_Number,
      X_Device_Country,
      X_Device_Sales_Model,
      DEVICE_NAME,
      app_id,
      TIMESTAMPDIFF(SECOND, prev_log_create_time, log_create_time) AS duration_sec
  FROM ordered
  WHERE visible = false
    AND prev_visible = true
    AND prev_log_create_time IS NOT NULL
    AND TIMESTAMPDIFF(SECOND, prev_log_create_time, log_create_time) <= 600
),

durations AS (
  SELECT
      platform,
      mac_addr,
      X_User_Number,
      X_Device_Country,
      X_Device_Sales_Model,
      DEVICE_NAME,
      CASE
        WHEN app_id = 'com.disney.disneyplus-prod' THEN 'disney+'
        WHEN app_id = 'youtube.leanback.v4' THEN 'youtube'
        WHEN app_id = 'amazon' THEN 'prime video'
        WHEN app_id = 'com.webos.app.livetv' THEN 'live_tv'
        WHEN app_id = 'com.webos.app.home' THEN 'home'
        WHEN app_id = 'com.webos.app.lgchannels' THEN 'lg_channels'
        WHEN app_id = 'netflix' THEN 'netflix'
        WHEN app_id LIKE 'com.webos.app.hdmi%' THEN 'hdmi'
        ELSE app_id
      END AS app,
      duration_sec
  FROM durations_raw
),

pre_entry AS (
  SELECT
      platform,
      mac_addr,
      X_User_Number,
      X_Device_Country,
      X_Device_Sales_Model,
      DEVICE_NAME,
      log_create_time,
      CASE
        WHEN app_id = 'com.disney.disneyplus-prod' THEN 'disney+'
        WHEN app_id = 'youtube.leanback.v4' THEN 'youtube'
        WHEN app_id = 'amazon' THEN 'prime video'
        WHEN app_id = 'com.webos.app.livetv' THEN 'live_tv'
        WHEN app_id = 'com.webos.app.home' THEN 'home'
        WHEN app_id = 'com.webos.app.lgchannels' THEN 'lg_channels'
        WHEN app_id = 'netflix' THEN 'netflix'
        WHEN app_id LIKE 'com.webos.app.hdmi%' THEN 'hdmi'
        ELSE app_id
      END AS app
  FROM base
  WHERE visible = true
),

entry AS (
  SELECT
      platform,
      mac_addr,
      X_User_Number,
      X_Device_Country,
      X_Device_Sales_Model,
      DEVICE_NAME,
      app,
      COUNT(*) AS entry_cnt,
      COUNT(DISTINCT DATE_FORMAT(log_create_time, 'yyyy-MM-dd')) AS active_days
  FROM pre_entry
  GROUP BY
      platform,
      mac_addr,
      X_User_Number,
      X_Device_Country,
      X_Device_Sales_Model,
      DEVICE_NAME,
      app
),

device_level_summary AS (
  SELECT
      e.platform,
      e.mac_addr,
      e.X_User_Number,
      e.X_Device_Country,
      e.X_Device_Sales_Model,
      e.DEVICE_NAME,
      e.app,
      e.entry_cnt,
      e.active_days,
      COUNT(d.duration_sec) AS completed_session_cnt,
      AVG(d.duration_sec) AS avg_duration_sec,
      SUM(d.duration_sec) AS total_duration_sec,
      SUM(CASE WHEN d.duration_sec <= 5 THEN 1 ELSE 0 END) AS bounce_5s_cnt,
      SUM(CASE WHEN d.duration_sec <= 10 THEN 1 ELSE 0 END) AS bounce_10s_cnt
  FROM entry e
  LEFT JOIN durations d
    ON e.platform = d.platform
   AND e.mac_addr = d.mac_addr
   AND COALESCE(e.X_User_Number, '') = COALESCE(d.X_User_Number, '')
   AND e.X_Device_Country = d.X_Device_Country
   AND e.X_Device_Sales_Model = d.X_Device_Sales_Model
   AND e.DEVICE_NAME = d.DEVICE_NAME
   AND e.app = d.app
  GROUP BY
      e.platform,
      e.mac_addr,
      e.X_User_Number,
      e.X_Device_Country,
      e.X_Device_Sales_Model,
      e.DEVICE_NAME,
      e.app,
      e.entry_cnt,
      e.active_days
)

SELECT
    platform,
    DEVICE_NAME,
    app,
    COUNT(*) AS device_cnt,
    ROUND(AVG(entry_cnt), 2) AS avg_entry_cnt,
    ROUND(AVG(active_days), 2) AS avg_active_days,
    ROUND(AVG(completed_session_cnt), 2) AS avg_completed_session_cnt,
    ROUND(AVG(avg_duration_sec), 2) AS avg_duration_sec,
    ROUND(AVG(total_duration_sec), 2) AS avg_total_duration_sec,
    ROUND(AVG(CASE WHEN completed_session_cnt > 0 THEN bounce_5s_cnt * 1.0 / completed_session_cnt END), 4) AS avg_bounce_5s_rate,
    ROUND(AVG(CASE WHEN completed_session_cnt > 0 THEN bounce_10s_cnt * 1.0 / completed_session_cnt END), 4) AS avg_bounce_10s_rate
FROM device_level_summary
GROUP BY platform, DEVICE_NAME, app
ORDER BY platform, DEVICE_NAME, app;
